In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

In [ ]:
#
# Given:
#   logits : Float[batch, posn, d_vocab]   raw scores from the transformer
#   tokens : Int  [batch, posn]            actual token IDs in the sequence
#
# Returns:
#   Float[batch, posn-1]   log P(correct next token) at every position

def get_log_probs(logits, tokens):
    log_probs = logits.log_softmax(dim=-1)                          # step 1
    log_probs_for_tokens = (
        log_probs[:, :-1]                                           # step 2
        .gather(dim=-1, index=tokens[:, 1:].unsqueeze(-1))          # step 3 + 4
        .squeeze(-1)                                                 # step 5
    )
    return log_probs_for_tokens

In [ ]:
#
# Sequence: ["The", "cat", "sat", "."]   token IDs: [1, 2, 3, 4]
# 1 batch item, 4 positions, tiny vocab of 6 tokens.

BATCH   = 1
POSN    = 4      # sequence length
D_VOCAB = 6      # vocabulary size

# Simulate logits: random unnormalised scores the model might produce.
logits = torch.randn(BATCH, POSN, D_VOCAB)

# Ground-truth token IDs.
#   pos 0 → "The"  (1)
#   pos 1 → "cat"  (2)
#   pos 2 → "sat"  (3)
#   pos 3 → "."    (4)
tokens = torch.tensor([[1, 2, 3, 4]])   # shape [1, 4]

print("logits shape :", logits.shape)   # [1, 4, 6]
print("tokens shape :", tokens.shape)   # [1, 4]
print()
print("logits:\n", logits)
print()
print("tokens:\n", tokens)

logits shape : torch.Size([1, 4, 6])
tokens shape : torch.Size([1, 4])

logits:
 tensor([[[-1.1258, -1.1524, -0.2506, -0.4339,  0.8487,  0.6920],
         [-0.3160, -2.1152,  0.4681, -0.1577,  1.4437,  0.2660],
         [ 0.1665,  0.8744, -0.1435, -0.1116,  0.9318,  1.2590],
         [ 2.0050,  0.0537,  0.6181, -0.4128, -0.8411, -2.3160]]])

tokens:
 tensor([[1, 2, 3, 4]])


In [ ]:
#
# logits[b, p, :] is a vector of D_VOCAB raw scores for position p in batch b.
# The scores are unnormalised — they can be any real number and do not sum to 1.
#
# ── From a single logit vector to a probability distribution ─────────────────
#
# Softmax first turns the raw scores into probabilities:
#
#   P(token_v | context) = exp(logits[b, p, v])
#                          ─────────────────────────────────────────
#                          sum over all v' of exp(logits[b, p, v'])
#
#   The denominator sums exp(score) across the entire vocabulary,
#   so every entry is non-negative and the whole vector sums to 1.
#
# Taking the log of both sides gives log-softmax:
#
#   log P(token_v | context) = log(  exp(logits[b, p, v])
#                                    ─────────────────────────────── )
#                                    sum_v' exp(logits[b, p, v'])
#
#                            = log exp(logits[b, p, v])
#                              - log( sum_v' exp(logits[b, p, v']) )
#
#                            = logits[b, p, v]
#                              - log( sum_v' exp(logits[b, p, v']) )
#
#   The second step uses  log(a/b) = log(a) - log(b).
#   The third step uses   log(exp(x)) = x.
#   The subtracted term is a single scalar (the log-normaliser) that is the
#   same for every v — it does not depend on v, only on the full score vector
#   at position p for batch item b.
#
# ── What "context up to position p" means ───────────────────────────────────
#
#   A causal transformer at position p has seen tokens at positions 0 … p
#   (it cannot attend to future positions).  Its hidden state at position p
#   therefore encodes the entire left context, and the logit vector
#   logits[b, p, :] is the model's output distribution *given* that context:
#   "what token is most likely to come next?".
#
#   So  log_probs[b, p, v]  = log P(next token = v | tokens[b, 0], …, tokens[b, p])
#
# ── dim=-1 ───────────────────────────────────────────────────────────────────
#
#   The normalising sum runs over all vocabulary entries, which live on the
#   last axis (dim=-1, size D_VOCAB).  Using dim=-1 applies the operation
#   independently for every (b, p) pair — i.e. one separate softmax per
#   position per batch item — without touching the batch or position axes.
#   The shape is unchanged: [batch, posn, d_vocab].
#
# ── Why log-softmax instead of softmax + log ────────────────────────────────
#
#   For likely next tokens the raw probability can be very close to 1, but
#   for unlikely tokens it can be so small (e.g. 1e-40) that float32 rounds
#   it to 0 before the log is taken, producing -inf.  log_softmax computes
#   the same result algebraically without ever materialising tiny probabilities,
#   so it is numerically stable.

log_probs = logits.log_softmax(dim=-1)

print("log_probs shape:", log_probs.shape)   # still [1, 4, 6]

# Sanity check: exp(log_probs) along vocab axis must sum to 1 at every position.
prob_sums = log_probs.exp().sum(dim=-1)
print("exp(log_probs).sum(dim=-1) [should be all 1.0]:\n", prob_sums)

log_probs shape: torch.Size([1, 4, 6])
exp(log_probs).sum(dim=-1) [should be all 1.0]:
 tensor([[1.0000, 1.0000, 1.0000, 1.0000]])


In [ ]:
#
# SHAPE BEFORE: [batch, posn,   d_vocab]  =  [1, 4, 6]
# SHAPE AFTER : [batch, posn-1, d_vocab]  =  [1, 3, 6]
#
# The index expression  [:, :-1]  contains only TWO slots, but the tensor
# has THREE dimensions.  PyTorch (like NumPy) applies a simple rule:
#
#   Any trailing dimension not mentioned gets an implicit ':'
#   (meaning "keep every element along that axis").
#
# So  [:, :-1]  is identical to  [:, :-1, :]  — explicitly written:
#
#   dim 0  →  :      keep all batch items                (unchanged)
#   dim 1  →  :-1    keep positions 0 … posn-2, DROP the last position
#   dim 2  →  :      keep ALL vocabulary entries         (implicit, unchanged)
#
# Think of log_probs as a stack of (batch) matrices each of shape
# [posn × d_vocab].  The slice removes only the LAST ROW of each matrix
# while leaving every row's full width (d_vocab columns) completely intact.
#
# Why drop the last position?
#   The transformer at position (posn-1) produces a prediction, but there is
#   no ground-truth *next* token to compare it against, so it cannot contribute
#   to the loss and is simply discarded here.

lp_sliced = log_probs[:, :-1]          # identical to log_probs[:, :-1, :]

print("log_probs shape  :", log_probs.shape)   # [1, 4, 6]
print("lp_sliced shape  :", lp_sliced.shape)   # [1, 3, 6]
print()

# Show explicitly that [:, :-1] and [:, :-1, :] are the same tensor.
assert torch.equal(log_probs[:, :-1], log_probs[:, :-1, :]), "should be identical"
print("log_probs[:, :-1] == log_probs[:, :-1, :] ✓")
print()

# Inspect: which rows were kept, which was dropped?
print("log_probs (all positions):\n",  log_probs[0])    # 4 rows × 6 cols
print()
print("log_probs[:, :-1] (positions 0-2 only):\n", lp_sliced[0])   # 3 rows × 6 cols
print()
print("Dropped row (position 3):\n", log_probs[0, -1])  # the row that was removed

log_probs shape  : torch.Size([1, 4, 6])
lp_sliced shape  : torch.Size([1, 3, 6])

log_probs[:, :-1] == log_probs[:, :-1, :] ✓

log_probs (all positions):
 tensor([[-2.9823, -3.0088, -2.1070, -2.2903, -1.0077, -1.1644],
        [-2.4955, -4.2947, -1.7114, -2.3372, -0.7358, -1.9134],
        [-2.2672, -1.5593, -2.5771, -2.5453, -1.5018, -1.1746],
        [-0.4398, -2.3911, -1.8267, -2.8576, -3.2859, -4.7608]])

log_probs[:, :-1] (positions 0-2 only):
 tensor([[-2.9823, -3.0088, -2.1070, -2.2903, -1.0077, -1.1644],
        [-2.4955, -4.2947, -1.7114, -2.3372, -0.7358, -1.9134],
        [-2.2672, -1.5593, -2.5771, -2.5453, -1.5018, -1.1746]])

Dropped row (position 3):
 tensor([-0.4398, -2.3911, -1.8267, -2.8576, -3.2859, -4.7608])


In [ ]:
#
# SHAPE BEFORE: [batch, posn]    =  [1, 4]
# SHAPE AFTER : [batch, posn-1]  =  [1, 3]
#
# Index expression  [:, 1:]  maps to:
#
#   dim 0  →  :    keep all batch items
#   dim 1  →  1:   keep indices 1, 2, …, posn-1  — DROP index 0
#
# General Python slice  a:b  keeps every index i where  a <= i < b.
# Omitting b means "go to the end of the axis".
# So  1:  keeps {1, 2, 3, …} and discards only index 0 (the first token).
#
# Effect: LEFT-SHIFT by one position.
#   result[b, i]  ==  tokens[b, i+1]   for every b, i
#
#   original tokens:  ["The"(1), "cat"(2), "sat"(3), "."(4)]
#   after  [:, 1:]:               ["cat"(2), "sat"(3), "."(4)]
#                                    ↑ index 0   ↑ index 1   ↑ index 2
#
# Why drop the first token?
#   "The" (index 0) is the very first token — no model position exists that
#   should have predicted it as a *next* token, so it has no role as a target.
#   The remaining tokens are the ground-truth TARGETS:
#     - tokens[:, 1:][b, 0] = "cat"  → what position 0 ("The") should predict
#     - tokens[:, 1:][b, 1] = "sat"  → what position 1 ("cat") should predict
#     - tokens[:, 1:][b, 2] = "."    → what position 2 ("sat") should predict

tok_sliced = tokens[:, 1:]

print("tokens shape    :", tokens.shape)       # [1, 4]
print("tok_sliced shape:", tok_sliced.shape)   # [1, 3]
print()
print("tokens    :", tokens)       # [[1, 2, 3, 4]]
print("tok_sliced:", tok_sliced)   # [[   2, 3, 4]]

tokens shape    : torch.Size([1, 4])
tok_sliced shape: torch.Size([1, 3])

tokens    : tensor([[1, 2, 3, 4]])
tok_sliced: tensor([[2, 3, 4]])


In [ ]:
#
# After the two slices:
#
#   lp_sliced   shape [batch, posn-1, d_vocab]  = [1, 3, 6]
#   tok_sliced  shape [batch, posn-1]            = [1, 3]
#
# Position i in lp_sliced  → the distribution the model output at step i
# Position i in tok_sliced → the token the model SHOULD have predicted at step i
#
#   lp_sliced [b, 0, :]  ←→  tok_sliced[b, 0]  ("cat")   target for pos 0
#   lp_sliced [b, 1, :]  ←→  tok_sliced[b, 1]  ("sat")   target for pos 1
#   lp_sliced [b, 2, :]  ←→  tok_sliced[b, 2]  (".")     target for pos 2
#
# The last log_prob row (pos 3) and the first token (index 0) were each
# discarded — they are the unmatched ends with no counterpart.

for i, (pos_name, tgt_name) in enumerate(
    [(0, "cat"), (1, "sat"), (2, ".")]
):
    print(f"  pos {pos_name}: distribution over {D_VOCAB} tokens → "
          f"target = '{tgt_name}' (id={tok_sliced[0, i].item()})")

  pos 0: distribution over 6 tokens → target = 'cat' (id=2)
  pos 1: distribution over 6 tokens → target = 'sat' (id=3)
  pos 2: distribution over 6 tokens → target = '.' (id=4)


In [ ]:
#
# torch.gather(input, dim, index) requires:
#   index.shape == input.shape   (same number of dimensions)
#
# lp_sliced  has 3 dims: [batch, posn-1, d_vocab]
# tok_sliced has 2 dims: [batch, posn-1]
#
# unsqueeze(-1) inserts a size-1 dimension at the last axis:
#   [batch, posn-1]  →  [batch, posn-1, 1]
#
# The value 1 means "we want to pick exactly one entry along the vocab axis
# for every (batch, posn) slot" — which is exactly what gather will do.

idx = tok_sliced.unsqueeze(-1)

print("tok_sliced shape:", tok_sliced.shape)   # [1, 3]
print("idx shape       :", idx.shape)          # [1, 3, 1]
print()
print("idx:\n", idx)

tok_sliced shape: torch.Size([1, 3])
idx shape       : torch.Size([1, 3, 1])

idx:
 tensor([[[2],
         [3],
         [4]]])


In [ ]:
#
# gather(dim=-1, index=idx) reads, for each coordinate (b, p, 0):
#
#   output[b, p, 0] = lp_sliced[b, p, idx[b, p, 0]]
#                   = lp_sliced[b, p, tokens[b, p+1]]
#                   = log P(true next token | context up to position p)
#
# Concretely for batch item 0:
#   output[0, 0, 0] = lp_sliced[0, 0, 2]   ← log P("cat" | "The")
#   output[0, 1, 0] = lp_sliced[0, 1, 3]   ← log P("sat" | "The cat")
#   output[0, 2, 0] = lp_sliced[0, 2, 4]   ← log P("."   | "The cat sat")
#
# Shape after gather: [batch, posn-1, 1]  (the vocab axis collapsed to size 1)

gathered = lp_sliced.gather(dim=-1, index=idx)

print("lp_sliced shape:", lp_sliced.shape)   # [1, 3, 6]
print("gathered shape :", gathered.shape)    # [1, 3, 1]
print()
print("gathered:\n", gathered)

# Verify manually: each value should equal the entry at the target token index.
for p in range(POSN - 1):
    target_id   = tok_sliced[0, p].item()
    manual_val  = lp_sliced[0, p, target_id].item()
    gather_val  = gathered[0, p, 0].item()
    print(f"  pos {p}: target id={target_id}  "
          f"manual={manual_val:.4f}  gather={gather_val:.4f}  match={manual_val == gather_val}")

lp_sliced shape: torch.Size([1, 3, 6])
gathered shape : torch.Size([1, 3, 1])

gathered:
 tensor([[[-2.1070],
         [-2.3372],
         [-1.5018]]])
  pos 0: target id=2  manual=-2.1070  gather=-2.1070  match=True
  pos 1: target id=3  manual=-2.3372  gather=-2.3372  match=True
  pos 2: target id=4  manual=-1.5018  gather=-1.5018  match=True


In [ ]:
#
# gathered has shape [batch, posn-1, 1].
# The trailing 1 was only there to satisfy gather's dimension requirement.
# squeeze(-1) removes it, yielding the clean output shape [batch, posn-1].

result = gathered.squeeze(-1)

print("gathered shape:", gathered.shape)   # [1, 3, 1]
print("result shape  :", result.shape)     # [1, 3]
print()
print("result:", result)
print()
print("Interpretation:")
print(f"  log P('cat' | 'The')         = {result[0, 0].item():.4f}")
print(f"  log P('sat' | 'The cat')     = {result[0, 1].item():.4f}")
print(f"  log P('.'   | 'The cat sat') = {result[0, 2].item():.4f}")

gathered shape: torch.Size([1, 3, 1])
result shape  : torch.Size([1, 3])

result: tensor([[-2.1070, -2.3372, -1.5018]])

Interpretation:
  log P('cat' | 'The')         = -2.1070
  log P('sat' | 'The cat')     = -2.3372
  log P('.'   | 'The cat sat') = -1.5018


In [ ]:
output = get_log_probs(logits, tokens)

print("get_log_probs output:", output)
print("manual result       :", result)
print()
assert torch.allclose(output, result), "mismatch!"
print("Both approaches give identical results ✓")

get_log_probs output: tensor([[-2.1070, -2.3372, -1.5018]])
manual result       : tensor([[-2.1070, -2.3372, -1.5018]])

Both approaches give identical results ✓


In [ ]:
#
# Nothing in the logic is specific to a single sequence.
# Every slice and gather operates independently along dim 0.

BATCH2 = 3
logits2 = torch.randn(BATCH2, POSN, D_VOCAB)
tokens2 = torch.randint(0, D_VOCAB, (BATCH2, POSN))

output2 = get_log_probs(logits2, tokens2)

print("logits2 shape :", logits2.shape)    # [3, 4, 6]
print("tokens2 shape :", tokens2.shape)   # [3, 4]
print("output2 shape :", output2.shape)   # [3, 3]  ← batch=3, posn-1=3
print()
print("output2:\n", output2)

# Each row is one batch item; each column is one prediction position.
# output2[b, p] = log P(tokens2[b, p+1] | tokens2[b, :p+1])

logits2 shape : torch.Size([3, 4, 6])
tokens2 shape : torch.Size([3, 4])
output2 shape : torch.Size([3, 3])

output2:
 tensor([[-2.4797, -3.4928, -2.4943],
        [-2.8981, -2.0227, -1.7429],
        [-1.5696, -1.5048, -2.5014]])


In [ ]:
#
# The standard language-modelling loss is the mean of the negated log probs:
#
#   L = - (1 / (posn-1)) * sum_p  log P(x_{p+1} | x_{<=p})
#
# That is just -mean(get_log_probs(logits, tokens)).

log_probs_per_token = get_log_probs(logits2, tokens2)   # [3, 3]

# Mean over all positions and all batch items.
loss_manual = -log_probs_per_token.mean()

# Cross-entropy via F.cross_entropy for comparison.
# F.cross_entropy expects shape [N, C] for logits and [N] for targets.
# We flatten (batch × posn-1) into N and use the sliced logits/tokens.
lp_flat  = logits2[:, :-1].reshape(-1, D_VOCAB)    # [(batch*(posn-1)), d_vocab]
tgt_flat = tokens2[:, 1:].reshape(-1)               # [(batch*(posn-1))]
loss_ce  = F.cross_entropy(lp_flat, tgt_flat)

print(f"manual loss   : {loss_manual.item():.6f}")
print(f"F.cross_entropy: {loss_ce.item():.6f}")
assert torch.isclose(loss_manual, loss_ce, atol=1e-5), "losses should match"
print("Losses match ✓")

manual loss   : 2.300691
F.cross_entropy: 2.300691
Losses match ✓
